# Session 14 Solutions: Project 1: A Complete Python Program

Data Science with Python, Level 1, Stage 1: Python Foundations

Try each exercise yourself before reading its solution. There is usually more than one correct answer; compare your approach with this one.

## In-Class Exercises

### In-Class Exercise 1: Choose and Plan

In [1]:
# Example plan for project B, the budget tracker:
# Data: each entry has a date, a description, a category and an amount (negative for expenses).
# Actions: add an entry, list entries, report by month, report by category, save and quit.
# Rules: amounts must be numbers; dates must be in the form YYYY-MM-DD; categories come from a fixed list.
# Files: entries.csv with one entry per line.
print("Plan written")

Plan written


### In-Class Exercise 2: The Data Class

In [2]:
class Entry:
    CATEGORIES = ["Food", "Transport", "Books", "Salary", "Other"]

    def __init__(self, date, description, category, amount):
        if category not in Entry.CATEGORIES:
            raise ValueError(f"unknown category {category}")
        self.date, self.description = date, description
        self.category, self.amount = category, float(amount)

    def month(self):
        return self.date[:7]

    def __str__(self):
        return f"{self.date}  {self.description:<18}{self.category:<10}{self.amount:>9.2f}"

print(Entry("2025-03-01", "Salary", "Salary", 900))
print(Entry("2025-03-02", "Bus card", "Transport", -15))

2025-03-01  Salary            Salary       900.00
2025-03-02  Bus card          Transport    -15.00


### In-Class Exercise 3: Save and Load

In [3]:
class Entry:
    def __init__(self, date, description, category, amount):
        self.date, self.description, self.category, self.amount = date, description, category, float(amount)

def save(entries, filename):
    with open(filename, "w") as f:
        f.write("date,description,category,amount\n")
        for e in entries:
            f.write(f"{e.date},{e.description},{e.category},{e.amount}\n")

def load(filename):
    with open(filename) as f:
        f.readline()
        return [Entry(*line.strip().split(",")) for line in f]

entries = [Entry("2025-03-01", "Salary", "Salary", 900), Entry("2025-03-02", "Bus card", "Transport", -15)]
save(entries, "entries.csv")
loaded = load("entries.csv")
print(len(loaded), loaded[1].description, loaded[1].amount)

2 Bus card -15.0


### In-Class Exercise 4: The Menu

In [4]:
def ask_float(prompt):
    while True:
        try:
            return float(input(prompt))
        except ValueError:
            print("Please type a number.")

def menu():
    total = 0
    while True:
        choice = input("1 Add  2 Total  3 Quit: ")
        if choice == "1":
            total += ask_float("Amount: ")
        elif choice == "2":
            print("Total:", total)
        elif choice == "3":
            break
        else:
            print("Unknown option")

menu()

1 Add  2 Total  3 Quit: 1
Amount: 2
1 Add  2 Total  3 Quit: python
Unknown option
1 Add  2 Total  3 Quit: 3


## Part A: Basic

### Exercise 1: Milestone 1: The Plan

In [5]:
# Project A: Campus Bookshop
# Data: Book(title, price, stock, category); the shop keeps a dict of books by title.
# Actions: list, search, sell, restock, report, save and quit.
# Rules: no sale beyond the stock; 10 percent off for 10 or more copies; 15 percent tax.
# Files: catalogue.csv (title,price,stock,category) and sales.csv (title,copies,paid).
print("Plan ready")

Plan ready


### Exercise 2: Milestone 2: The Data Class

In [6]:
class Book:
    TAX_RATE = 0.15

    def __init__(self, title, price, stock, category):
        if float(price) < 0 or int(stock) < 0:
            raise ValueError("price and stock cannot be negative")
        self.title, self.category = title, category
        self.price, self.stock = float(price), int(stock)

    def sell(self, quantity):
        if quantity <= 0 or quantity > self.stock:
            raise ValueError(f"cannot sell {quantity}; {self.stock} in stock")
        rate = 0.10 if quantity >= 10 else 0
        self.stock -= quantity
        return round(self.price * quantity * (1 - rate) * (1 + Book.TAX_RATE), 2)

    def restock(self, copies):
        if copies <= 0:
            raise ValueError("copies must be positive")
        self.stock += copies

    def __str__(self):
        return f"{self.title:<26}{self.category:<13}{self.price:>7.2f}{self.stock:>6}"

b1 = Book("Python Basics", 45.0, 12, "Programming")
b2 = Book("SQL in a Week", 29.9, 7, "Programming")
print(b1)
print(b2)
try:
    Book("Bad Book", -1, 3, "Data")
except ValueError as e:
    print("Refused:", e)

Python Basics             Programming    45.00    12
SQL in a Week             Programming    29.90     7
Refused: price and stock cannot be negative


## Part B: Intermediate

### Exercise 3: Milestone 3: Files

In [7]:
class Book:
    def __init__(self, title, price, stock, category):
        self.title, self.category = title, category
        self.price, self.stock = float(price), int(stock)

def save_catalogue(books, filename):
    with open(filename, "w") as f:
        f.write("title,price,stock,category\n")
        for b in books.values():
            f.write(f"{b.title},{b.price},{b.stock},{b.category}\n")

def load_catalogue(filename):
    books = {}
    try:
        with open(filename) as f:
            f.readline()
            for line in f:
                t, p, s, c = line.strip().split(",")
                books[t] = Book(t, p, s, c)
    except FileNotFoundError:
        print("No catalogue found, starting empty")
    return books

books = {"Python Basics": Book("Python Basics", 45.0, 12, "Programming"),
         "SQL in a Week": Book("SQL in a Week", 29.9, 7, "Programming")}
save_catalogue(books, "catalogue.csv")
again = load_catalogue("catalogue.csv")
print([(b.title, b.stock) for b in again.values()])
print(load_catalogue("missing.csv"))

[('Python Basics', 12), ('SQL in a Week', 7)]
No catalogue found, starting empty
{}


### Exercise 4: Milestone 4: The Menu

In [8]:
def ask_int(prompt):
    while True:
        try:
            return int(input(prompt))
        except ValueError:
            print("Please type a whole number.")

stock = {"Python Basics": 12, "SQL in a Week": 7}

def menu():
    while True:
        choice = input("1 List  2 Sell  3 Quit: ").strip()
        if choice == "1":
            for title, n in stock.items():
                print(f"{title:<20}{n:>4}")
        elif choice == "2":
            title = input("Title: ").strip()
            if title not in stock:
                print("No such book")
                continue
            n = ask_int("Copies: ")
            if 0 < n <= stock[title]:
                stock[title] -= n
                print("Sold", n)
            else:
                print("Sale refused")
        elif choice == "3":
            print("Goodbye")
            break
        else:
            print("Unknown option")

menu()

1 List  2 Sell  3 Quit: Python Basics
Unknown option
1 List  2 Sell  3 Quit: 2
Title: y
No such book
1 List  2 Sell  3 Quit: 3
Goodbye


## Part C: Challenge

### Exercise 5: Milestone 5: Tests

In [9]:
class Book:
    def __init__(self, title, price, stock):
        self.title, self.price, self.stock = title, float(price), int(stock)

    def sell(self, quantity):
        if quantity <= 0 or quantity > self.stock:
            raise ValueError("impossible sale")
        rate = 0.10 if quantity >= 10 else 0
        self.stock -= quantity
        return round(self.price * quantity * (1 - rate) * 1.15, 2)

b = Book("Test", 10.0, 30)
assert b.sell(1) == 11.5
assert b.sell(9) == 103.5
assert b.sell(10) == 103.5
assert b.stock == 10
for bad in [0, -2, 11]:
    try:
        b.sell(bad)
        raise AssertionError(f"sale of {bad} was accepted")
    except ValueError:
        pass
print("All tests passed")
# Bug found while testing: selling 0 copies used to be accepted; fixed with quantity <= 0.

All tests passed


## Part D: Cumulative Project: Campus Bookshop

### Exercise 6: Milestone 6: README and Submission

In [10]:
example = '''
1 List  2 Search  3 Sell  4 Report  5 Save and quit
Choose: 3
Title: Python Basics
Copies: 2
Paid 103.50.
Choose: 4
Revenue this session: 103.50
Out of stock: ['Statistics Made Simple']
'''
print(example)


1 List  2 Search  3 Sell  4 Report  5 Save and quit
Choose: 3
Title: Python Basics
Copies: 2
Paid 103.50.
Choose: 4
Revenue this session: 103.50
Out of stock: ['Statistics Made Simple']

